# Neural Network Regularization: Dropout vs L2 (Weight Decay)

## Overview
This notebook trains a **deliberately over-parameterized** Feedforward Neural Network (FFNN) on MNIST in **three configurations** to make the effect of regularization clearly visible:

| Variant | Dropout | L2 (Weight Decay) |
|---------|---------|-------------------|
| **Baseline** (no regularization) | ✗ | ✗ |
| **Dropout** | ✓ (p = 0.5) | ✗ |
| **L2 Regularization** | ✗ | ✓ (λ = 1e-3) |

**Why over-parameterized?** A small network on MNIST is already near-optimal, so regularization effects are subtle. By using a very large network (1.5 M+ parameters) with only 40k training samples and training for 30 full epochs, the baseline will clearly overfit, making the comparison dramatic and illustrative.

**Goal:** Compare training and testing accuracies across the three variants and discuss how regularization techniques help improve model generalization.

---
### ⚡ 2× T4 GPU Optimizations applied
- `nn.DataParallel` splits each batch across both T4s
- Normalization baked into `transforms` (runs on GPU-prefetched tensors, no per-batch CPU work)
- Batch size doubled to **1024** (fills both GPUs)
- `num_workers=4` + `persistent_workers=True` for async CPU → GPU data loading
- **Mixed Precision (AMP)** with `autocast` + `GradScaler` — cuts memory use and speeds up Tensor Core math
- `optimizer.zero_grad(set_to_none=True)` — skips zeroing and just frees grad memory
- All GPU CUDA seeds set for reproducibility

In [ ]:
# Imports & Setup
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
import torchvision.datasets as datasets
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import warnings
import copy
warnings.filterwarnings('ignore')

# Reproducibility
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)
    torch.cuda.manual_seed_all(42)   # covers all GPUs (both T4s)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
n_gpus = torch.cuda.device_count()
print(f'Using device : {device}')
print(f'GPUs available: {n_gpus}')
for i in range(n_gpus):
    print(f'  GPU {i}: {torch.cuda.get_device_name(i)}')

## 1. Data Loading

We use **40,000 training samples** (instead of the full 48k) to stress the network more and make overfitting more visible. Validation uses 20,000 samples and the test set has 10,000.

**Dual-GPU note:** Batch size is doubled to 1024 so both T4s get a 512-sample chunk each via `DataParallel`. `num_workers=4` and `persistent_workers=True` keep the CPU → GPU pipeline saturated.

In [ ]:
# Data Loading
# Normalization is baked into the transform so it runs on already-pinned tensors
# rather than inside the training loop on the CPU.
MNIST_MEAN, MNIST_STD = 0.1307, 0.3081

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((MNIST_MEAN,), (MNIST_STD,)),
])

data_root = './data'
full_train    = datasets.MNIST(root=data_root, train=True,  transform=transform, download=True)
test_dataset  = datasets.MNIST(root=data_root, train=False, transform=transform, download=True)

# Split: 40k train / 20k validation
train_dataset, val_dataset = random_split(
    full_train, [40000, 20000],
    generator=torch.Generator().manual_seed(42)
)

# Doubled batch size for 2× T4 (each GPU sees 512 samples per step)
batch_size   = 1024
pin_memory   = torch.cuda.is_available()
num_workers  = 4   # async CPU worker processes

loader_kwargs = dict(
    batch_size=batch_size,
    pin_memory=pin_memory,
    num_workers=num_workers,
    persistent_workers=(num_workers > 0),  # keep workers alive between epochs
)

train_loader = DataLoader(train_dataset, shuffle=True,  **loader_kwargs)
val_loader   = DataLoader(val_dataset,   shuffle=False, **loader_kwargs)
test_loader  = DataLoader(test_dataset,  shuffle=False, **loader_kwargs)

print(f'Training   samples: {len(train_dataset)}')
print(f'Validation samples: {len(val_dataset)}')
print(f'Test       samples: {len(test_dataset)}')
print(f'Batch size         : {batch_size}  ({batch_size // max(n_gpus,1)} per GPU)')

## 2. Model Architecture

We use a **very wide, deep network** so that the baseline model has far more capacity than needed for MNIST.

```
Input (784)
  → Linear(784 → 1024) → ReLU → [Dropout?]
  → Linear(1024 → 512) → ReLU → [Dropout?]
  → Linear(512 → 256)  → ReLU → [Dropout?]
  → Linear(256 → 128)  → ReLU → [Dropout?]
  → Linear(128 → 10)   → Output
```

**~1.6 million parameters** — far more than necessary for 40k samples of 28×28 images.

| Variant | Dropout (p) | Weight Decay (λ) |
|---------|-------------|------------------|
| Baseline | 0.0 | 0 |
| Dropout | 0.5 | 0 |
| L2 | 0.0 | 5e-4 |

**Dual-GPU:** each model is wrapped in `nn.DataParallel`, which automatically splits every batch across both T4s and gathers gradients before the optimizer step.

In [ ]:
# Model Definition

class FFNN(nn.Module):
    """Deliberately over-parameterized FFNN with optional Dropout."""
    def __init__(self, dropout_rate=0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, 1024), nn.ReLU(), nn.Dropout(dropout_rate),
            nn.Linear(1024, 512), nn.ReLU(), nn.Dropout(dropout_rate),
            nn.Linear(512, 256),  nn.ReLU(), nn.Dropout(dropout_rate),
            nn.Linear(256, 128),  nn.ReLU(), nn.Dropout(dropout_rate),
            nn.Linear(128, 10),
        )

    def forward(self, x):
        return self.net(x)


# Three configurations
model_configs = {
    'Baseline (No Reg)': {'dropout': 0.0, 'weight_decay': 0.0},
    'Dropout (p=0.5)':   {'dropout': 0.5, 'weight_decay': 0.0},
    'L2 (λ=5e-4)':       {'dropout': 0.0, 'weight_decay': 5e-4},
}

models = {}
for name, cfg in model_configs.items():
    m = FFNN(dropout_rate=cfg['dropout']).to(device)
    # Wrap in DataParallel if more than one GPU is present
    if n_gpus > 1:
        m = nn.DataParallel(m)
    models[name] = m
    n_params = sum(p.numel() for p in m.parameters())
    dp_tag = f'DataParallel({n_gpus}×T4)' if n_gpus > 1 else 'SingleGPU'
    print(f'{name:22s}  params={n_params:,}  dropout={cfg["dropout"]}  wd={cfg["weight_decay"]}  [{dp_tag}]')

In [ ]:
# Training & Evaluation Helpers

scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

def train_one_epoch(model, loader, loss_fn, optimizer):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)          # faster than zeroing

        with torch.cuda.amp.autocast():                # fp16 forward pass
            outputs = model(images)
            loss    = loss_fn(outputs, labels)

        scaler.scale(loss).backward()                  # scaled backprop
        scaler.step(optimizer)                         # unscale + step
        scaler.update()                                # adjust scale factor

        running_loss += loss.item() * labels.size(0)
        correct      += (outputs.argmax(1) == labels).sum().item()
        total        += labels.size(0)
    return running_loss / total, 100.0 * correct / total


@torch.no_grad()
def evaluate(model, loader, loss_fn):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        with torch.cuda.amp.autocast():
            outputs = model(images)
            loss    = loss_fn(outputs, labels)
        running_loss += loss.item() * labels.size(0)
        correct      += (outputs.argmax(1) == labels).sum().item()
        total        += labels.size(0)
    return running_loss / total, 100.0 * correct / total


print('Helpers defined  (AMP enabled, set_to_none=True, non_blocking transfers).')

## 3. Training

All three models train for the multiple **epochs** without early stopping. This lets the baseline model overfit progressively, while regularized models hold their validation performance.

In [ ]:
# Training Loop
NUM_EPOCHS = 30
LR = 1e-3
loss_fn = nn.CrossEntropyLoss()

histories = {}

for name, cfg in model_configs.items():
    print(f'\n{"="*65}')
    print(f'  Training: {name}')
    print(f'{"="*65}')

    model = models[name]
    optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=cfg['weight_decay'])
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

    for epoch in range(1, NUM_EPOCHS + 1):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, loss_fn, optimizer)
        vl_loss, vl_acc = evaluate(model, val_loader, loss_fn)

        history['train_loss'].append(tr_loss)
        history['val_loss'].append(vl_loss)
        history['train_acc'].append(tr_acc)
        history['val_acc'].append(vl_acc)

        print(f'  Ep {epoch:2d}/{NUM_EPOCHS}  '
              f'Train Loss={tr_loss:.4f} Acc={tr_acc:6.2f}%  '
              f'Val Loss={vl_loss:.4f} Acc={vl_acc:6.2f}%')

    histories[name] = history

print(f'\n{"="*65}')
print('All models trained.')

## 4. Training Curves

Because there is no early stopping, all 30 epochs are visible. Watch the baseline's **validation loss rise** while training loss keeps falling — the hallmark of overfitting. Regularized models should keep their validation curves much flatter.

In [ ]:
# Plot Style (Light Theme)
plt.rcParams.update({
    'figure.facecolor': '#ffffff',
    'axes.facecolor':   '#f6f8fa',
    'axes.edgecolor':   '#d0d7de',
    'axes.labelcolor':  '#24292f',
    'text.color':       '#24292f',
    'xtick.color':      '#57606a',
    'ytick.color':      '#57606a',
    'grid.color':       '#d0d7de',
    'legend.facecolor': '#ffffff',
    'legend.edgecolor': '#d0d7de',
    'font.family':      'sans-serif',
    'font.size':        11,
})

# Train color = solid, Val color = lighter/hatched visually distinct
STYLES = {
    'Baseline (No Reg)': {'train': '#c0392b', 'val': '#e74c3c'},
    'Dropout (p=0.5)':   {'train': '#1a6faf', 'val': '#3498db'},
    'L2 (λ=5e-4)':       {'train': '#1e8449', 'val': '#27ae60'},
}

print('Plot style configured.')

In [ ]:
# Loss Curves — All Three Models Together (easy to compare)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Training vs Validation Loss and Accuracy (30 Epochs, No Early Stopping)',
             fontsize=14, fontweight='bold')

ax_loss, ax_acc = axes

for name, hist in histories.items():
    c = STYLES[name]
    ep = range(1, len(hist['train_loss']) + 1)
    ax_loss.plot(ep, hist['train_loss'], '-',  color=c['train'], linewidth=2.5, label=f'{name} Train')
    ax_loss.plot(ep, hist['val_loss'],   '--', color=c['val'],   linewidth=2,   label=f'{name} Val')
    ax_acc.plot( ep, hist['train_acc'], '-',  color=c['train'], linewidth=2.5, label=f'{name} Train')
    ax_acc.plot( ep, hist['val_acc'],   '--', color=c['val'],   linewidth=2,   label=f'{name} Val')

ax_loss.set_title('Loss', fontsize=13, fontweight='bold')
ax_loss.set_xlabel('Epoch')
ax_loss.set_ylabel('Loss')
ax_loss.legend(fontsize=9, ncol=2)
ax_loss.grid(True, alpha=0.4)

ax_acc.set_title('Accuracy (%)', fontsize=13, fontweight='bold')
ax_acc.set_xlabel('Epoch')
ax_acc.set_ylabel('Accuracy (%)')
ax_acc.legend(fontsize=9, ncol=2)
ax_acc.grid(True, alpha=0.4)

plt.tight_layout()
plt.savefig('loss_acc_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: loss_acc_curves.png')

In [ ]:
# Per-Model Loss Subplots — Easier to see each model's gap
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Per-Model: Training vs Validation Loss (Overfitting Visible)',
             fontsize=14, fontweight='bold')

for ax, (name, hist) in zip(axes, histories.items()):
    c = STYLES[name]
    ep = range(1, len(hist['train_loss']) + 1)
    ax.plot(ep, hist['train_loss'], '-',  color=c['train'], linewidth=2.5, label='Train Loss')
    ax.plot(ep, hist['val_loss'],   '--', color=c['val'],   linewidth=2,   label='Val Loss')
    ax.fill_between(ep, hist['train_loss'], hist['val_loss'],
                    alpha=0.12, color=c['train'], label='Overfit gap')
    ax.set_title(name, fontsize=13, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.4)

plt.tight_layout()
plt.savefig('loss_per_model.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: loss_per_model.png')

In [ ]:
# Per-Model Accuracy Subplots — Train vs Val gap
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Per-Model: Training vs Validation Accuracy (Overfitting Visible)',
             fontsize=14, fontweight='bold')

for ax, (name, hist) in zip(axes, histories.items()):
    c = STYLES[name]
    ep = range(1, len(hist['train_acc']) + 1)
    ax.plot(ep, hist['train_acc'], '-',  color=c['train'], linewidth=2.5, label='Train Acc')
    ax.plot(ep, hist['val_acc'],   '--', color=c['val'],   linewidth=2,   label='Val Acc')
    ax.fill_between(ep, hist['val_acc'], hist['train_acc'],
                    alpha=0.12, color=c['train'], label='Overfit gap')
    ax.set_title(name, fontsize=13, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Accuracy (%)')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.4)

plt.tight_layout()
plt.savefig('acc_per_model.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: acc_per_model.png')

## 5. Test Set Evaluation

In [ ]:
# Test Evaluation
results = []

for name in model_configs:
    model = models[name]
    te_loss, te_acc = evaluate(model, test_loader, loss_fn)
    tr_acc = histories[name]['train_acc'][-1]   # final epoch train acc
    vl_acc = histories[name]['val_acc'][-1]     # final epoch val acc
    best_vl_acc = max(histories[name]['val_acc'])
    gap = tr_acc - te_acc

    results.append({
        'Model': name,
        'Train Acc (%)': round(tr_acc, 2),
        'Best Val Acc (%)': round(best_vl_acc, 2),
        'Test Acc (%)':  round(te_acc, 2),
        'Gap (Train-Test)': round(gap, 2),
    })

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

In [ ]:
# Train vs Test Accuracy Bar Chart
model_names = results_df['Model'].tolist()
train_accs  = results_df['Train Acc (%)'].tolist()
test_accs   = results_df['Test Acc (%)'].tolist()

fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(model_names))
width = 0.32

bars1 = ax.bar(x - width/2, train_accs, width, label='Train Accuracy (final epoch)',
               color=[STYLES[n]['train'] for n in model_names], alpha=0.85)
bars2 = ax.bar(x + width/2, test_accs,  width, label='Test Accuracy',
               color=[STYLES[n]['val']   for n in model_names], alpha=0.85)

for bar in list(bars1) + list(bars2):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_ylabel('Accuracy (%)', fontsize=12)
ax.set_title('Training vs Test Accuracy: Regularization Comparison', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(model_names, fontsize=11)
ax.legend(fontsize=11)
ax.set_ylim(80, 102)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('accuracy_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: accuracy_comparison.png')

In [ ]:
# Generalization Gap Bar Chart
gaps = results_df['Gap (Train-Test)'].tolist()

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(model_names, gaps,
              color=[STYLES[n]['train'] for n in model_names],
              alpha=0.85, width=0.5)

for bar, gap in zip(bars, gaps):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            f'{gap:.2f}%', ha='center', va='bottom', fontsize=12, fontweight='bold')

ax.set_ylabel('Generalization Gap (%)', fontsize=12)
ax.set_title('Generalization Gap (Train Acc - Test Acc)\nLarger = More Overfitting',
             fontsize=13, fontweight='bold')
ax.axhline(y=0, color='#57606a', linestyle='--', linewidth=0.8)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('generalization_gap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: generalization_gap.png')

## 6. Overfitting Analysis

The most illustrative plot: the **gap between train and val accuracy over all 30 epochs**. The baseline's gap should grow consistently, while Dropout and L2 models keep theirs small.

In [ ]:
# Overfitting Gap Over Epochs (Train - Val Accuracy)
fig, ax = plt.subplots(figsize=(11, 6))

for name, hist in histories.items():
    gap = [t - v for t, v in zip(hist['train_acc'], hist['val_acc'])]
    ep  = range(1, len(gap) + 1)
    ax.plot(ep, gap, '-o', color=STYLES[name]['train'],
            markersize=4, linewidth=2.5, label=name)

ax.axhline(y=0, color='#57606a', linestyle='--', linewidth=1)
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Train Acc - Val Acc (%)', fontsize=12)
ax.set_title('Overfitting Gap Over Epochs\n(Higher = More Overfit; Regularized Models Should Stay Near 0)',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.4)

plt.tight_layout()
plt.savefig('overfitting_gap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: overfitting_gap.png')

## 7. Confusion Matrices

In [ ]:
# Confusion Matrices
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.suptitle('Confusion Matrices (Test Set)', fontsize=16, fontweight='bold')

for ax, name in zip(axes, model_configs):
    model = models[name]
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            with torch.cuda.amp.autocast():
                preds = model(images).argmax(1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    cm = confusion_matrix(all_labels, all_preds)
    cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True) * 100

    sns.heatmap(cm_norm, annot=True, fmt='.1f', cmap='Blues', ax=ax,
                xticklabels=range(10), yticklabels=range(10),
                cbar_kws={'label': 'Recall (%)'})
    ax.set_title(name, fontsize=13, fontweight='bold')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')

plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: confusion_matrices.png')

## 8. Summary Dashboard

In [ ]:
# Summary Dashboard
fig = plt.figure(figsize=(20, 14))
fig.suptitle('Regularization Comparison: Summary Dashboard (30 Epochs, No Early Stopping)',
             fontsize=16, fontweight='bold', y=0.99)

# Row 1: Loss and Accuracy overlaid
ax1 = fig.add_subplot(2, 3, 1)
ax2 = fig.add_subplot(2, 3, 2)

for name, hist in histories.items():
    c = STYLES[name]
    ep = range(1, len(hist['train_loss']) + 1)
    ax1.plot(ep, hist['train_loss'], '-',  color=c['train'], linewidth=2, label=f'{name[:8]} Tr')
    ax1.plot(ep, hist['val_loss'],   '--', color=c['val'],   linewidth=1.5, label=f'{name[:8]} Va')
    ax2.plot(ep, hist['train_acc'], '-',  color=c['train'], linewidth=2, label=f'{name[:8]} Tr')
    ax2.plot(ep, hist['val_acc'],   '--', color=c['val'],   linewidth=1.5, label=f'{name[:8]} Va')

ax1.set_title('Loss (All Models)', fontsize=12, fontweight='bold')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.legend(fontsize=7, ncol=2); ax1.grid(True, alpha=0.4)

ax2.set_title('Accuracy (All Models)', fontsize=12, fontweight='bold')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy (%)')
ax2.legend(fontsize=7, ncol=2); ax2.grid(True, alpha=0.4)

# Overfitting gap
ax3 = fig.add_subplot(2, 3, 3)
for name, hist in histories.items():
    gap = [t - v for t, v in zip(hist['train_acc'], hist['val_acc'])]
    ax3.plot(range(1, len(gap)+1), gap, '-o', color=STYLES[name]['train'],
             markersize=3, linewidth=2, label=name)
ax3.axhline(0, color='grey', linestyle='--', linewidth=0.8)
ax3.set_title('Overfit Gap (Train - Val Acc)', fontsize=12, fontweight='bold')
ax3.set_xlabel('Epoch'); ax3.set_ylabel('Gap (%)')
ax3.legend(fontsize=9); ax3.grid(True, alpha=0.4)

# Train vs Test bar
ax4 = fig.add_subplot(2, 3, 4)
x = np.arange(len(model_names)); w = 0.32
ax4.bar(x - w/2, train_accs, w, label='Train',
        color=[STYLES[n]['train'] for n in model_names], alpha=0.85)
ax4.bar(x + w/2, test_accs,  w, label='Test',
        color=[STYLES[n]['val']   for n in model_names], alpha=0.85)
for i, (tr, te) in enumerate(zip(train_accs, test_accs)):
    ax4.text(i-w/2, tr+0.3, f'{tr:.1f}', ha='center', fontsize=8, fontweight='bold')
    ax4.text(i+w/2, te+0.3, f'{te:.1f}', ha='center', fontsize=8, fontweight='bold')
ax4.set_title('Train vs Test Accuracy', fontsize=12, fontweight='bold')
ax4.set_xticks(x); ax4.set_xticklabels(model_names, fontsize=9)
ax4.set_ylim(80, 102); ax4.legend(); ax4.grid(axis='y', alpha=0.3)

# Generalization gap bar
ax5 = fig.add_subplot(2, 3, 5)
gaps = results_df['Gap (Train-Test)'].tolist()
brs = ax5.bar(model_names, gaps,
              color=[STYLES[n]['train'] for n in model_names], alpha=0.85, width=0.5)
for b, g in zip(brs, gaps):
    ax5.text(b.get_x()+b.get_width()/2, b.get_height()+0.1,
             f'{g:.2f}%', ha='center', fontsize=10, fontweight='bold')
ax5.set_title('Generalization Gap', fontsize=12, fontweight='bold')
ax5.set_ylabel('Gap (%)')
ax5.axhline(0, color='grey', linestyle='--', linewidth=0.8)
ax5.grid(axis='y', alpha=0.3)

# Summary text
ax6 = fig.add_subplot(2, 3, 6)
ax6.axis('off')
summary = (
    'KEY FINDINGS\n'
    '─────────────────────────────\n\n'
    'Baseline:\n'
    '  Train acc keeps rising,\n'
    '  val/test acc plateaus or drops.\n'
    '  Largest overfit gap.\n\n'
    'Dropout (p=0.5):\n'
    '  Train acc is suppressed.\n'
    '  Val/test acc matches closely.\n'
    '  Smallest overfit gap.\n\n'
    'L2 (λ=5e-4):\n'
    '  Moderate weight penalty.\n'
    '  Smoother train/val curves.\n'
    '  Small overfit gap.\n'
)
ax6.text(0.05, 0.95, summary, transform=ax6.transAxes,
         fontsize=11, verticalalignment='top', fontfamily='monospace',
         bbox=dict(boxstyle='round', facecolor='#f6f8fa', alpha=0.8))

plt.tight_layout()
plt.savefig('summary_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: summary_dashboard.png')

## 9. Discussion: How Regularization Improves Generalization

### What Is Overfitting?
Overfitting occurs when a neural network memorizes the training data (including noise) rather than learning generalizable patterns. A model that overfits achieves **high training accuracy** but **poor test accuracy**, where the gap between the two is the **generalization gap**.

In this notebook the effect is amplified deliberately:
- The network has ~1.6 M parameters, far more than MNIST needs.
- Only 40,000 training samples are used.
- All 30 epochs are run without stopping.

This combination guarantees that the **baseline clearly overfits**, so the benefit of regularization is unmistakable.

### Baseline (No Regularization)
- Training loss decreases every epoch and training accuracy approaches 100%.
- Validation loss starts rising after the first several epochs — the network is memorizing, not generalizing.
- The generalization gap (Train Acc - Test Acc) is the **largest** among the three models.

### Dropout Regularization (p = 0.5)
- **How it works:** During each forward pass in training, Dropout randomly **zeroes out 50% of neurons** in each hidden layer. This forces the network to learn **redundant, distributed representations** rather than relying on any single neuron or co-adaptation of neurons.
- **Effect on training:** Training accuracy is **lower** than the baseline because the model is effectively training a different sub-network each iteration.
- **Effect on testing:** At test time, all neurons are active (with scaled weights), effectively **ensembling** many sub-networks. This produces **better generalization** and a **smaller gap** between training and test accuracy.
- **Intuition:** Dropout acts as an implicit ensemble of $2^n$ thinned networks (where $n$ is the number of dropout-eligible neurons).

### L2 Regularization (Weight Decay, λ = 5e-4)
- **How it works:** L2 regularization adds a penalty term $\frac{\lambda}{2} \|\mathbf{w}\|_2^2$ to the loss function. This **discourages large weights** by shrinking them toward zero.
- **Effect:** The network prefers **simpler, smoother** decision boundaries because large weights (which create sharp, complex boundaries) are penalized.
- **Result:** Training accuracy is slightly lower than the unregularized baseline, but the test accuracy improves because the model's decision surface generalizes better to unseen data.
- **Mathematically:** The gradient update becomes:
  $$w \leftarrow w - \eta \left( \nabla_{w} \mathcal{L} + \lambda w \right) = (1 - \eta \lambda) w - \eta \nabla_{w} \mathcal{L}$$
  This $(1 - \eta \lambda)$ factor is why L2 regularization is called **weight decay**.

### Summary

| Aspect | Baseline | Dropout | L2 Reg |
|--------|----------|---------|--------|
| Training Accuracy | **Highest** (near 100%) | Lower | Slightly Lower |
| Test Accuracy | Lowest | **Higher** | **Higher** |
| Generalization Gap | **Largest** | **Smallest** | Small |
| Mechanism | None | Random neuron masking | Weight magnitude penalty |
| Effect | Memorizes data | Learns robust features | Prefers simple models |

**Key Takeaway:** Regularization techniques reduce overfitting by constraining the model's capacity. Dropout forces distributed representations through random masking, while L2 penalizes weight magnitudes to favor simpler models. Both lead to **better generalization** (i.e., improved test accuracy relative to training accuracy).

In [ ]:
# Classification Reports
for name in model_configs:
    model = models[name]
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            with torch.cuda.amp.autocast():
                preds = model(images).argmax(1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    print(f'\n{"="*60}')
    print(f'Classification Report: {name}')
    print(f'{"="*60}')
    print(classification_report(all_labels, all_preds, digits=4))

In [ ]:
print('\nNotebook complete. Saved plots:')
for f in ['loss_acc_curves.png', 'loss_per_model.png', 'acc_per_model.png',
          'accuracy_comparison.png', 'generalization_gap.png',
          'overfitting_gap.png', 'confusion_matrices.png', 'summary_dashboard.png']:
    print(f'  {f}')